In [16]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr
import sys
sys.path.insert(0, '/home/codycruz/LLC_ocean_emulator/high_res_diagnostics')
from predictions_io import open_predictions
import cftime
import os
import json
from scipy.spatial import cKDTree

In [17]:
# ============== LOAD LLC ==============

# DOMAIN: face = int for one face, slice(0,13) for the globe (cache faces 7-12 are north-up, like the emulator)
face=1#slice(0,13)#1
x_extent=slice(0,270)
y_extent=slice(0,270)

# Coarse cache: normalized float16 channels (U_0..Salt_50, Eta), land 0 -> physical units, land NaN
cache = xr.open_zarr('/orcd/scratch/seedfund/001/llc_emulator/coarse/LLC4320_coarse_cache_0p33deg.zarr',
                     consolidated=False).drop_vars('boundary')
native = zarr.open_group(cache.attrs['native_store'], mode='r')
stats = {s: xr.DataArray(np.concatenate([native[f'{v}_{s}'][:] for v in ('U', 'V', 'Theta', 'Salt', 'Eta')]),
                         dims='channel') for s in ('mean', 'std')}
prog = (cache.prognostic * stats['std'] + stats['mean']).where(cache.wet.astype(bool))
llc_vars = {v: prog.isel(channel=slice(51 * n, 51 * (n + 1))).rename(channel='k').transpose('time', 'k', 'face', 'y', 'x')
            for n, v in enumerate(['U', 'V', 'Theta', 'Salt'])}
llc_vars['Eta'] = prog.isel(channel=204)

llc_patch_full_face = xr.Dataset({**llc_vars, 'XC': cache.XC, 'YC': cache.YC, 'rA': cache.rA}).assign_coords(
    face=np.arange(13), k=np.arange(51)
).isel(
    face=face,
    x=x_extent,
    y=y_extent
)

In [18]:
x_extent=slice(180,225)
y_extent=slice(45,90)

llc_patch_full = llc_patch_full_face.isel(
    x=x_extent,
    y=y_extent
)

# ============== LOAD EMULATORS ==============
emulator_configs = [
    {
        'name': 'w384-ckpt46',
        'key': 'emulator_1',
        'path': '/orcd/scratch/seedfund/001/llc_emulator/inference/coarse/w384-ckpt46/predictions.zarr',
        'desc': ''
    },
#     {
#         'name': 'face-1-ckpt4',
#         'key': 'emulator_2',
#         'path': '/orcd/scratch/seedfund/001/llc_emulator/inference/1face/epoch4-ema-quintic-reg/predictions.zarr',
#         'desc': ''
#     },
#     {
#         'name': 'band_h-ckpt4',
#         'key': 'emulator_3',
#         'path': '/orcd/scratch/seedfund/001/llc_emulator/inference/1face//epoch4-ema-quintic-band_h/predictions.zarr',
#         'desc': ''
#     },
#    {
#         'name': 'band_hz-ckpt4',
#         'key': 'emulator_4',
#         'path': '/orcd/scratch/seedfund/001/llc_emulator/inference/1face//epoch4-ema-quintic-band_hz/predictions.zarr',
#         'desc': ''
#     },
]

# ============== OPEN EMULATOR DATASETS ==============
emulator_patches_raw = {}
for cfg in emulator_configs:
    # Load emulator data
    emu_ds = open_predictions(cfg['path'])
    
    # Rename dimensions to match LLC if needed
    # (adjust these names based on your actual emulator dimension names)
    dim_mapping = {}
    if 'lat' in emu_ds.dims and 'y' not in emu_ds.dims:
        dim_mapping['lat'] = 'y'
    if 'lon' in emu_ds.dims and 'x' not in emu_ds.dims:
        dim_mapping['lon'] = 'x'
    
    if dim_mapping:
        emu_ds = emu_ds.rename(dim_mapping)
    
    # Apply same spatial slicing as LLC
    emu_ds = emu_ds.isel(face=face, x=x_extent, y=y_extent)
    
    # Align emulator to LLC's coordinates
    emu_ds = emu_ds.assign_coords({
        'x': llc_patch_full.x,
        'y': llc_patch_full.y
    })
    
    emulator_patches_raw[cfg['key']] = emu_ds
    print(f"Loaded {cfg['name']}: {cfg['desc']}")

# ============== TIME MATCHING ==============
def normalize_times(times):
    return pd.DatetimeIndex([
        pd.Timestamp(
            int(t.year), int(t.month), int(t.day),
            int(t.hour), int(t.minute), int(t.second)
        )
        if hasattr(t, "year")
        else pd.Timestamp(t).floor("s")
        for t in times
    ])

llc_times_norm = normalize_times(llc_patch_full.time.values)

common_times = llc_times_norm
for cfg in emulator_configs:
    emulator_times_norm = normalize_times(emulator_patches_raw[cfg['key']].time.values)
    common_times = common_times.intersection(emulator_times_norm)

common_times = common_times.sort_values()

llc_mask = llc_times_norm.isin(common_times)
llc_patch = llc_patch_full.isel(time=llc_mask)

print(f"LLC subset to {len(common_times)} common times")

# ============== PORT GRID VARS & BUILD UNIFIED STRUCTURE ==============
grid_vars = ['XC', 'YC', 'rA', 'Z', 'dxC', 'dyC', 'dxG', 'dyG']

emulator_patches = {}
for cfg in emulator_configs:
    patch_raw = emulator_patches_raw[cfg['key']]
    patch_times_norm = normalize_times(patch_raw.time.values)

    patch_mask = patch_times_norm.isin(common_times)
    patch = patch_raw.isel(time=patch_mask)

    # Add LLC grid variables
    for gv in grid_vars:
        if gv in llc_patch:
            patch[gv] = llc_patch[gv]

    emulator_patches[cfg['key']] = patch

# ============== UNIFIED REFERENCE LISTS ==============
emulator_info = [(cfg['name'], cfg['key']) for cfg in emulator_configs]
n_emulators = len(emulator_info)

all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"\n=== Setup complete: LLC + {n_emulators} emulators ===")
for name, key in emulator_info:
    print(f"  {name} ({key})")
    print(f"    Shape: {emulator_patches[key][list(emulator_patches[key].data_vars)[0]].shape}")
    print(f"    x range: [{emulator_patches[key].x.values.min()}, {emulator_patches[key].x.values.max()}]")
    print(f"    y range: [{emulator_patches[key].y.values.min()}, {emulator_patches[key].y.values.max()}]")

print(f"LLC shape: {llc_patch[list(llc_patch.data_vars)[0]].shape}")

Loaded w384-ckpt46: 
LLC subset to 744 common times

=== Setup complete: LLC + 1 emulators ===
  w384-ckpt46 (emulator_1)
    Shape: (744, 51, 45, 45)
    x range: [0, 44]
    y range: [0, 44]
LLC shape: (744, 51, 45, 45)


In [19]:
selected_time_range = [0, 744]   # inclusive indices
stepping = 48                    # 1 = every timestep, 4 = every 4th timestep

start_idx, end_idx = selected_time_range

# ----------------------------------------
# First subset LLC
# ----------------------------------------
llc_patch = llc_patch.isel(
    time=slice(start_idx, end_idx + 1, stepping)
)

# ----------------------------------------
# Then subset each emulator safely
# Handles shorter emulator runs automatically
# ----------------------------------------
emulator_patches_subset = {}

for key, patch in emulator_patches.items():

    max_time = patch.sizes['time']

    # Prevent indexing past emulator length
    safe_end_idx = min(end_idx, max_time - 1)

    patch_subset = patch.isel(
        time=slice(start_idx, safe_end_idx + 1, stepping)
    )

    emulator_patches_subset[key] = patch_subset

emulator_patches = emulator_patches_subset

# ----------------------------------------
# Match LLC length to shortest emulator
# ----------------------------------------
min_time_len = min(
    [llc_patch.sizes['time']] +
    [patch.sizes['time'] for patch in emulator_patches.values()]
)

llc_patch = llc_patch.isel(time=slice(0, min_time_len))

emulator_patches = {
    key: patch.isel(time=slice(0, min_time_len))
    for key, patch in emulator_patches.items()
}

# ----------------------------------------
# Rebuild combined dict
# ----------------------------------------
all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

# ----------------------------------------
# Diagnostics
# ----------------------------------------
print(f"Subset to time indices {start_idx}:{end_idx}")
print(f"Stepping = {stepping}")
print(f"Final synchronized length = {min_time_len}")

print(f"LLC now has {llc_patch.sizes['time']} times")

for name, key in emulator_info:
    print(
        f"{name} ({key}) now has "
        f"{emulator_patches[key].sizes['time']} times"
    )

Subset to time indices 0:744
Stepping = 48
Final synchronized length = 16
LLC now has 16 times
w384-ckpt46 (emulator_1) now has 16 times


In [20]:
def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"
# ============== VERTICAL-AXIS HELPERS (W lives on cell interfaces) ==============
# LLC keeps W on `k_p1` -- 52 cell INTERFACES -- while Theta/Salt/U/V sit on `k`,
# 51 cell CENTRES. The emulator writes W back onto `k` with 51 levels. So a
# blanket `.isel(k=...)` silently leaves LLC's W at full depth (the level index
# never applies) and the plot then fails on a 3-D array, or a difference
# broadcasts 52 against 51. Level L means the same layer in both: the emulator's
# `k=L` is trained against LLC's `k_p1=L`.
VERTICAL_DIMS = ("k", "k_p1", "k_l", "k_u", "lev")


def vertical_dim_of(da):
    """Whichever vertical dimension this variable carries, or None if surface."""
    for dim in VERTICAL_DIMS:
        if dim in da.dims:
            return dim
    return None


def pick(patch, var, *, time=None, k=None):
    """`patch[var]` with `time`/`k` applied to the axes it actually has."""
    da = patch[var]
    if time is not None:
        da = da.isel(time=time)
    if k is not None:
        dim = vertical_dim_of(da)
        if dim is not None:
            da = da.isel({dim: k})
    return da


def common_levels(*patch_var_pairs):
    """Levels the given (patch, var) pairs share.

    LLC's W has 52 and the emulator's has 51, so depth comparisons have to trim
    to the shorter of the two rather than broadcast.
    """
    sizes = []
    for patch, var in patch_var_pairs:
        dim = vertical_dim_of(patch[var])
        sizes.append(patch[var].sizes[dim] if dim is not None else 1)
    return min(sizes)


def depth_axis(patch, var):
    """Depth in metres for each level of `var`, negative downward.

    The store carries `Z` for cell centres but no coordinate for the interfaces
    W sits on, so derive them: face[0] = 0, face[i+1] = 2*centre[i] - face[i].
    That is the same construction as DEPTH_INTERFACES in ocean_emulators.
    """
    centres = np.asarray(patch['Z'].values, dtype=float)
    if vertical_dim_of(patch[var]) != 'k_p1':
        return centres
    faces = [0.0]
    for centre in centres:
        faces.append(2.0 * centre - faces[-1])
    return np.asarray(faces)

In [21]:
# ============== SET VARIABLES HERE ==============
prog_vars = ['Theta']#, 'Salt', 'U', 'V', 'Eta']
depth = 0  # k level
colormaps = {'Theta': 'Spectral_r', 'Salt': 'viridis', 'U': 'bwr', 'V': 'bwr', 'Eta': 'viridis'}
# ================================================

# Globe: nearest-neighbour regrid of all faces onto a 1/3 deg lon/lat grid (stitches seams, dateline, Arctic; land stays NaN).
# One face: plot the face as stored (north-up).
globe = 'face' in llc_patch.dims
if globe:
    def _xyz(lon, lat):
        lon, lat = np.deg2rad(lon), np.deg2rad(lat)
        return np.stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)], axis=-1)
    lon_g, lat_g = np.meshgrid(np.arange(-180, 180, 1 / 3) + 1 / 6, np.arange(-90, 90, 1 / 3) + 1 / 6)
    tree = cKDTree(_xyz(llc_patch.XC.values.ravel(), llc_patch.YC.values.ravel()))
    _, regrid_idx = tree.query(_xyz(lon_g.ravel(), lat_g.ravel()))
    to_plane = lambda da: da.transpose('face', 'y', 'x').values.ravel()[regrid_idx].reshape(lon_g.shape)
    extent, panel_size, tag = [-180, 180, -90, 90], (6, 3), 'globe'
else:
    to_plane = lambda da: da.transpose('y', 'x').values
    extent, panel_size, tag = None, (3.6, 3), f'face{face}'

os.makedirs('figs/simple/', exist_ok=True)
ref_patch = emulator_patches[emulator_info[0][1]]
n_times = len(ref_patch.time)
labels = ['LLC'] + [name for name, _ in emulator_info]
patches = [llc_patch] + [emulator_patches[key] for _, key in emulator_info]

for var in prog_vars:
    print(f"Generating plots for {var} k={depth} ({tag})...")
    ncols = len(patches)
    fig, axes = plt.subplots(n_times, ncols, figsize=(panel_size[0] * ncols, panel_size[1] * n_times),
                             dpi=200, squeeze=False)

    for row in range(n_times):
        time_str = format_time(ref_patch.time.values[row])
        fields = [to_plane(pick(p, var, time=row, k=depth)) for p in patches]
        if not any(np.isfinite(f).any() for f in fields):
            print(f"  Skipping {var} {time_str}: no finite values")
            continue
        vmin = min(np.nanmin(f) for f in fields)
        vmax = max(np.nanmax(f) for f in fields)

        for col, (field, label) in enumerate(zip(fields, labels)):
            ax = axes[row, col]
            im = ax.imshow(field, origin='lower', extent=extent, cmap=colormaps[var], vmin=vmin, vmax=vmax,
                           aspect='auto', interpolation='nearest')
            ax.set_title(f'{label} {var} k={depth} {time_str}', fontsize=8)
            ax.tick_params(labelsize=7)
            if globe:
                ax.set_ylabel('Latitude', fontsize=7)
                if row == n_times - 1:
                    ax.set_xlabel('Longitude', fontsize=7)

        fig.colorbar(im, ax=axes[row, :], pad=0.02, fraction=0.046)

    out = f'figs/simple/fields-{var}-k{depth}-{tag}.png'
    plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved {out}")

Generating plots for Theta k=0 (face1)...
✓ Saved figs/simple/fields-Theta-k0-face1.png
